In [23]:
# !pip install --upgrade pip
# !pip install gdown
# !pip install transformers
# !pip install datasets
# !pip install sentencepiece 
# !pip install lxml 
# !pip install keras-swa keras-nlp
!pip install tensorflow-addons

In [1]:
5

5

In [2]:
# !gdown 19z28zHN5fFbZYF7VvF9Z2FOvFbClgghB -O data/train/

# !gdown 1BvODRSUKDEJgx8edA4Su1qRhxm6zESM5 -O data/train/

# !gdown 1n4VbFABJs4HGJ-CuelqL1MLDTgE6I0zm -O data/train/

!gdown 1QOWKaPJ_reurIniuHFIpBhub9Ygi4UT7 -O data/train/


Downloading...
From (uriginal): https://drive.google.com/uc?id=1QOWKaPJ_reurIniuHFIpBhub9Ygi4UT7
From (redirected): https://drive.google.com/uc?id=1QOWKaPJ_reurIniuHFIpBhub9Ygi4UT7&confirm=t&uuid=ee09ce35-f919-43fa-9542-32db05301cb3
To: /kaggle/working/data/train/SAU_aljazeera_news_0.xml
100%|█████████████████████████████████████████| 356M/356M [00:03<00:00, 102MB/s]


In [3]:
import os
import pandas as pd
from tqdm import tqdm

train_path = 'data/train/'
for f in tqdm(os.listdir(train_path)):
    if '.csv' in f:
        continue
    path = os.path.join(train_path, f)
    data = pd.read_xml(path,)
    data = data[['article', 'summary']]
    data = data.rename(columns={"article": 'text', 'summary': 'target'})
    data.to_csv(f'{train_path}/{f.split(".")[0]}.csv', index=False)
    !rm $path

100%|██████████| 1/1 [00:20<00:00, 20.92s/it]


In [4]:
!ls data/train

SAU_aljazeera_news_0.csv


In [5]:
# !gdown 1SeS5RRlrd5fSA36xSThC_2_gLTX92_aL -O data/dev/


# !gdown 1cXZ3QxMVI3voLoORvHEc-MXl53EOm7tf -O data/dev/


!gdown 1p-s4h_i9BC0qlDuXXrFufKTBENz8q1pr -O data/dev/

Downloading...
From (uriginal): https://drive.google.com/uc?id=1p-s4h_i9BC0qlDuXXrFufKTBENz8q1pr
From (redirected): https://drive.google.com/uc?id=1p-s4h_i9BC0qlDuXXrFufKTBENz8q1pr&confirm=t&uuid=8760f15b-1992-4a97-be39-8f9347ebe868
To: /kaggle/working/data/dev/MUR_sahra_news.xml
100%|███████████████████████████████████████| 28.0M/28.0M [00:00<00:00, 124MB/s]


In [6]:
import os
import pandas as pd
from tqdm import tqdm

dev_path = 'data/dev/'
for f in tqdm(os.listdir(dev_path)):
    if '.csv' in f:
        continue
        
    path = os.path.join(dev_path, f)
    data = pd.read_xml(path)
    data = data[['article', 'summary']]
    data = data.rename(columns={"article": 'text', 'summary': 'target'})
    data.to_csv(f'{dev_path}/{f.split(".")[0]}.csv', index=False)
    !rm $path

100%|██████████| 1/1 [00:02<00:00,  2.85s/it]


In [7]:
!ls data/dev

MUR_sahra_news.csv


In [3]:
from dataclasses import dataclass
import numpy as np


class DataCollatorForSeq2Seq:
    def __init__(self, tokenizer, model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100, return_tensors='tf'):
        self.tokenizer =tokenizer
        self.model=model
        self.padding =padding
        self.max_length =max_length
        self.pad_to_multiple_of = pad_to_multiple_of
        self.label_pad_token_id = label_pad_token_id
        self.return_tensors = return_tensors

    def __call__(self, features, return_tensors=None):
        if return_tensors is None:
            return_tensors = self.return_tensors
        labels = [feature["labels"] for feature in features] if "labels" in features[0].keys() else None
        # We have to pad the labels before calling `tokenizer.pad` as this method won't pad them and needs them of the
        # same length to return tensors.
        if labels is not None:
            max_label_length = max(len(l) for l in labels)
            if self.pad_to_multiple_of is not None:
                max_label_length = (
                    (max_label_length + self.pad_to_multiple_of - 1)
                    // self.pad_to_multiple_of
                    * self.pad_to_multiple_of
                )

            padding_side = self.tokenizer.padding_side
            for feature in features:
                remainder = [self.label_pad_token_id] * (max_label_length - len(feature["labels"]))
                if isinstance(feature["labels"], list):
                    feature["labels"] = (
                        feature["labels"] + remainder if padding_side == "right" else remainder + feature["labels"]
                    )
                elif padding_side == "right":
                    feature["labels"] = np.concatenate([feature["labels"], remainder]).astype(np.int64)
                else:
                    feature["labels"] = np.concatenate([remainder, feature["labels"]]).astype(np.int64)

        features = self.tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=return_tensors,
        )

        # prepare decoder_input_ids
        if (
            labels is not None
            and self.model is not None
            and hasattr(self.model, "prepare_decoder_input_ids_from_labels")
        ):
            decoder_input_ids = self.model.prepare_decoder_input_ids_from_labels(labels=features["decoder_input_ids"])
            features["decoder_input_ids"] = decoder_input_ids

        return features

In [4]:
def agument(text):
    decoder_input = np.array(text.split())
    ss = len(decoder_input)
    r = np.random.random()

    if r > 0.85:
        np.random.shuffle(decoder_input)
    elif r > 0.7:
        
        idx = np.random.choice(np.arange(ss), size=int(0.8 * ss), replace=False).tolist()
        to_sh = decoder_input[idx]
        np.random.shuffle(to_sh)
        decoder_input[idx] = to_sh
    elif r > 0.55:


        idx = np.random.choice(np.arange(ss), size=int(0.5 * ss), replace=False).tolist()
        to_sh = decoder_input[idx]
        np.random.shuffle(to_sh)
        decoder_input[idx] = to_sh

    elif r > 0.4:

        idx = np.random.choice(np.arange(ss), size=int(0.3 * ss), replace=False).tolist()
        to_sh = decoder_input[idx]
        np.random.shuffle(to_sh)
        decoder_input[idx] = to_sh
    else:None
    
    return ' '.join(decoder_input.tolist())

def map_to(batch, train=True):
    model_inputs = tokenizer(batch['text'],
                              padding="max_length",
                              truncation=True,
                              max_length=900, return_tensors='np')

    decoder_input = batch['target']
    if train:
        decoder_input = list(map(agument, decoder_input))
    
    labels = tokenizer(decoder_input, text_target=batch['target'],
                       padding="max_length",
                       truncation=True,
                       max_length=300, return_tensors='np')
    
    model_inputs['decoder_input_ids'] = labels['input_ids']
    model_inputs['decoder_attention_mask'] = labels['attention_mask']
    model_inputs["labels"] = labels["labels"]
    
    return model_inputs


In [5]:
from functools import partial
import numpy as np

In [6]:
import tensorflow as tf

tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu="local") # "local" for 1VM TPU

strategy = tf.distribute.TPUStrategy(tpu)
print("RUNNING TPU")


print("REPLICAS: ", strategy.num_replicas_in_sync)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local
INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from arabert.preprocess import ArabertPreprocessor

model_name="abdalrahmanshahrour/arabartsummarization"
preprocessor = ArabertPreprocessor(model_name="")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipeline = pipeline("text2text-generation",model=model,tokenizer=tokenizer)

text = "شهدت مدينة طرابلس، مساء أمس الأربعاء، احتجاجات شعبية وأعمال شغب لليوم الثالث على التوالي، وذلك بسبب تردي الوضع المعيشي والاقتصادي. واندلعت مواجهات عنيفة وعمليات كر وفر ما بين الجيش اللبناني والمحتجين استمرت لساعات، إثر محاولة فتح الطرقات المقطوعة، ما أدى إلى إصابة العشرات من الطرفين."
text = preprocessor.preprocess(text)

result = pipeline(text,
            pad_token_id=tokenizer.eos_token_id,
            num_beams=3,
            repetition_penalty=3.0,
            max_length=200,
            length_penalty=1.0,
            no_repeat_ngram_size = 3)[0]['generated_text']
result
>>> "تجددت الاشتباكات بين الجيش اللبناني ومحتجين في مدينة طرابلس شمالي لبنان."


SyntaxError: invalid syntax (4213860981.py, line 22)

In [8]:
import tensorflow_addons as tfa
from datasets import load_dataset
from transformers import TFMT5ForConditionalGeneration, MT5Tokenizer
from tensorflow.keras.optimizers import AdamW
from keras_nlp.metrics import Perplexity
from swa.tfkeras import SWA



In [9]:
def set_model():
    
    
    with strategy.scope():

        model_name = "ahmeddbahaa/mt5-base-finetune-ar-xlsum"
        model_name = 'sebay/t5_ar'
        tokenizer = MT5Tokenizer.from_pretrained(model_name)
        
        model = TFMT5ForConditionalGeneration.from_pretrained(model_name) #, from_pt=True
        
#         swa = SWA(start_epoch=2, 
#                   lr_schedule='cyclic', 
#                   swa_lr=5e-6, 
#                   swa_lr2=1e-4,
#                   swa_freq=50,
#                   batch_size=24,
#                   verbose=1)
        optimizer = tf.keras.optimizers.legacy.Adam(5e-5,)
        opt = tfa.optimizers.SWA(optimizer, start_averaging=1,
                           average_period=50, lr=5e-4)
        model.compile(optimizer=opt, metrics=[Perplexity(from_logits=True, mask_token_id=tokenizer.pad_token_id, name="perplexity")])
        early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

        
        return model, [early_stopping], tokenizer

In [10]:
model, callbacks, tokenizer = set_model()

/usr/local/lib/python3.8/site-packages/keras/initializers/initializers.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
All model checkpoint layers were used when initializing TFMT5ForConditionalGeneration.

All the layers of TFMT5ForConditionalGeneration were initialized from the model checkpoint at sebay/t5_ar.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMT5ForConditionalGeneration for predictions without further training.
/usr/local/lib/python3.8/site-packages/tensorflow_addons/optimizers/average_wrapper.py:33: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [11]:
data = load_dataset('/kaggle/working/data',)

Found cached dataset csv (/root/.cache/huggingface/datasets/csv/data-2787d2c04b6ed045/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d)
100%|██████████| 2/2 [00:00<00:00, 388.70it/s]


In [12]:
data

DatasetDict({
    train: Dataset({
        features: ['text', 'target'],
        num_rows: 104883
    })
    validation: Dataset({
        features: ['text', 'target'],
        num_rows: 11983
    })
})

In [13]:
train_dataset = data['train'].shuffle(seed=40).map(map_to, batched=True, batch_size=2000, drop_last_batch=True, num_proc=25, remove_columns=['text', 'target'])
valid_dataset = data['validation'].shuffle(seed=40).map(partial(map_to, train=False), batched=True, batch_size=10, drop_last_batch=True, num_proc=25, remove_columns=['text', 'target'])

Loading cached shuffled indices for dataset at /root/.cache/huggingface/datasets/csv/data-2787d2c04b6ed045/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d/cache-cc72a2bb2f858342.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/data-2787d2c04b6ed045/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d/cache-faf2dbccca0c4f76_*_of_00025.arrow
Loading cached shuffled indices for dataset at /root/.cache/huggingface/datasets/csv/data-2787d2c04b6ed045/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d/cache-ed69c781970a22d9.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/data-2787d2c04b6ed045/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d/cache-3241283cc4b9bfaf_*_of_00025.arrow


In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors='tf')

In [15]:
tf_train_dataset = model.prepare_tf_dataset(
                                            train_dataset,
                                            collate_fn=data_collator,
                                            batch_size=24, drop_remainder=True,
                                            shuffle=True,
                                            )

tf_dev_dataset = model.prepare_tf_dataset(
                                        valid_dataset,
                                        collate_fn=data_collator,
                                        batch_size=24, drop_remainder=True,
                                        shuffle=False
                                        )

In [16]:
summary = model.fit(tf_train_dataset,validation_data=tf_dev_dataset, epochs=5, callbacks=callbacks,
                      initial_epoch=4) #validation_freq=2, workers=2, validation_steps=1000,

Epoch 5/5


/usr/local/lib/python3.8/site-packages/tensorflow/python/framework/indexed_slices.py:459: UserWarning: Converting sparse IndexedSlices to a dense Tensor with 192086016 elements. This may consume a large amount of memory.
  warnings.warn(
2023-07-06 00:07:13.558840: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-06 00:07:14.958426: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


4166/4166 [==============================] - ETA: 0s - loss: 0.5136 - perplexity: 20.6775

2023-07-06 00:33:39.020660: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-06 00:33:39.253302: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


4166/4166 [==============================] - 1733s 357ms/step - loss: 0.5136 - perplexity: 20.6775 - val_loss: 0.5771 - val_perplexity: 17.1862


In [20]:
!rm -r sebay/

In [21]:
model.save_pretrained("sebay/t5_ar",) 
tokenizer.save_pretrained("sebay/t5_ar") 

('sebay/t5_ar/tokenizer_config.json',
 'sebay/t5_ar/special_tokens_map.json',
 'sebay/t5_ar/spiece.model',
 'sebay/t5_ar/added_tokens.json')